# Example of Utility agent creation

Pre-requisites:

- `OPENAI_API_KEY` in a `.env` file
- Utility MCP server running


```
    cd langdmta_lab/mcps
    bash init_mcps.sh
```

In [1]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from dotenv import load_dotenv
from langdmta_lab.base.models import OpenAIGPT4oModel
from langgraph.prebuilt import create_react_agent

In [2]:
_ = load_dotenv()

In [10]:
client = MultiServerMCPClient(
    {
        "utility": {
            "transport": "streamable_http",
            "url": "http://127.0.0.1:8004/mcp"
        }
    }
)

tools = await client.get_tools()

In [11]:
llm = OpenAIGPT4oModel()

In [7]:
system_prompt = """
You are an agent with helper functions to validate if molecules are in their SMILES format and convert molecule names or compound numbers to SMILES.
  Your task is to address only the relevant parts of the question using the provided tools, no need to obtain confirmation from the supervisor. Always use exactly one tool per step, and only proceed to the next step after processing the result. If no tools are relevant, return control to the supervisor.
  If you used the synonym2smiles tool, always return the SMILES representations to the supervisor, no need to mention molecule names are invalid SMILES. When passing chemical names as input, you should not include any descriptors like "group, moiety, or substructure".
"""

In [ ]:
utility_agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt,
)

In [13]:
resp = await utility_agent.ainvoke(
    {"messages": [{"role": "user", "content": "What is the SMILES of Gleevec and amenamevir?"}]}
)
print(resp["messages"][-1].content)

The SMILES representation for Gleevec is "Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1", and for amenamevir, it is "Cc1cccc(C)c1N(CC(=O)Nc1ccc(-c2ncon2)cc1)C(=O)C1CCS(=O)(=O)CC1".
